In [0]:
%run ../01_Bronze/00_setup

In [0]:

# update the bin_event table by emptying the bins based on available employee pull
# update the inventory and shelves when the orders from bins are completed and shipped
# undate the orders when they have been completed (shipped)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

# 1. Identify bins in 'Shipment' status for the current batch
shipping_bins = (spark.table("workspace.amazon_fullfilment_silver.bins")
                 .filter("is_active = true")
                 .filter(F.col("status").isin("Shipment", "Picking")))

print(f"Total active Picking Bins: {shipping_bins.count()}")

# 2. Identify available Packers (1 bin per Packer constraint)
packers_df = (spark.table("workspace.amazon_fullfilment_silver.employee")
              .filter(f"is_active = true AND job_role = 'Packer' AND shift_id = '{current_shift}'")
              .select("employee_id")
              .withColumn("packer_rank", F.row_number().over(Window.orderBy("employee_id"))))

print(f"Total active Pakers: {packers_df.count()}") 

# 3. Assign Packers and Calculate Random Processing Time
# Formula for random minutes: 2 + (rand() * 8)
ranked_bins = shipping_bins.withColumn("bin_rank", F.row_number().over(Window.orderBy("bin_id")))

shipped_assignments = (ranked_bins.join(packers_df, ranked_bins.bin_rank == packers_df.packer_rank, "inner")
                       .withColumn("random_minutes", (F.lit(2) + (F.rand(seed=42) * 8)).cast("int"))
                       .withColumn("shipped_time", F.expr("updated_timestamp + make_interval(0, 0, 0, 0, 0, random_minutes, 0)")))

print(f"Total active shipped_assignments: {shipped_assignments.count()}") 

# --- OUTPUT A: Bin Records ---
bin_shipped_records = (shipped_assignments.select(
    "bin_id",
    "order_id",
    F.lit(0).alias("item_count"),
    F.lit("Shipped").alias("status"),
    "current_weight",
    packers_df["employee_id"], 
    F.col("shipped_time").alias("updated_timestamp"),
  #  F.lit(True).alias("is_active"),
  #  F.lit(current_run_uuid).alias("_batch_id"),
    F.current_timestamp().alias("_ingest_timestamp")
))

display(bin_shipped_records)

# --- OUTPUT B: Order Records ---
# 1. First, create the exploded list of unique order IDs from the assignments
exploded_shipped_orders = (shipped_assignments
    .withColumn("order_id", F.explode(F.split(F.col("order_id"), ", ")))
    .select("order_id", "shipped_time")
    .dropDuplicates(["order_id"]))

# 2. Join back to the original Bronze orders table to get the missing fields
order_shipped_records = (exploded_shipped_orders
    .join(
        spark.table("workspace.amazon_fullfilment_silver.orders_silver").select("order_id", "customer_id", "orderdate", "status", "is_current"),
        #spark.table("workspace.amazon_fullfilment_silver.orders_silver").select("order_id", "customer_id", "orderdate", "status","_batch_id"),  
        "order_id", 
        "inner"
    ).filter(F.col("status") == "Picking").filter("is_current == true")
    #).filter(F.col("status") == "Picking").filter(F.col("_batch_id") == current_run_uuid)
    .select(
        "order_id",
        "customer_id",
        "orderdate",
        F.lit("Shipped").alias("status"),
        F.col("shipped_time").alias("updated_timestamp")
       # "is_current",
     #   F.lit(current_run_uuid).alias("_batch_id"),
     #   F.current_timestamp().alias("_ingest_timestamp")
    ))

display(order_shipped_records)

### repeat the above for the inventory table.  Then you need to use the shelf_id associated to set the status of the shelf to 'empty'. Then independently you can reduce the robot battery to %2. Check if the battery is less than %10 and send to to charging.



# 1. Identify the Orders that were just Shipped
# We use the order_shipped_records from the previous step
shipped_order_ids = order_shipped_records.select("order_id")

# 2. Identify the Inventory items associated with those orders
# We need to know which shelf_id they came from before we remove them
current_inventory = spark.table("workspace.amazon_fullfilment_silver.inventory_silver")
#current_inventory = spark.table(f"{catalog_name}.{database_name}.inventory").filter(F.col("_batch_id") == current_run_uuid)

inventory_to_remove = current_inventory.join(shipped_order_ids, "order_id", "inner")

# --- OUTPUT A: Inventory "Pop" Records ---
# Since this is Bronze (Append-only), we create records with 0 or negative quantity 
# to represent the removal, or simply treat the absence of these IDs in Silver as the "Pop".
# For this simulation, we will generate the "Post-Removal" state.
new_inventory_state = current_inventory.join(shipped_order_ids, "order_id", "left_anti")

# 3. Determine Shelf Status (Idle vs Picking)
# A shelf is 'idle' ONLY if it no longer appears in the new_inventory_state
active_shelves = new_inventory_state.select(F.col("shelf_id").alias("active_shelf_id")).distinct()

# Get all shelves that were previously associated with our shipped orders
shelves_to_check = inventory_to_remove.select("shelf_id").distinct()

shelf_updates = (shelves_to_check.join(active_shelves, shelves_to_check.shelf_id == active_shelves.active_shelf_id, "left_outer")
    .withColumn("status", F.when(F.col("active_shelf_id").isNull(), "idle").otherwise("Picking"))
    # Join back to the original shelf table to get robot_id and capacity
    .join(spark.table("workspace.amazon_fullfilment_silver.shelves").select("shelf_id", "robot_id", "max_weight_capacity"), "shelf_id")
    .select(
        "shelf_id",
        "robot_id",
        "max_weight_capacity",
        "status",
        F.current_timestamp().alias("updated_timestamp")
      #  F.lit(True).alias("is_active"),
       # F.lit(current_run_uuid).alias("_batch_id"),
       # F.current_timestamp().alias("_ingest_timestamp")
    ))

# 4. Filter Inventory to only show what is LEAVING (The "Pop" action)
# In many Bronze architectures, you record the "Deletion" event
inventory_removal_records = (inventory_to_remove
    .withColumn("quantity", F.lit(0)) # Setting quantity to 0 indicates it's gone
    .select(
        "shelf_id",
        "product_id",
        "quantity",
        "order_id"
      #  F.lit(True).alias("is_active"),
        #F.lit(current_run_uuid).alias("_batch_id"),
       # F.current_timestamp().alias("_ingest_timestamp")
    ))

display(shelf_updates)
display(inventory_removal_records)
shelf_distinct_updates_df = shelf_updates.distinct()


# Save to source folder
(bin_shipped_records.write
 .format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/bin_events"))
(inventory_removal_records
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/inventory"))
(shelf_distinct_updates_df
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/shelves_events"))
(order_shipped_records
 .drop("_metadata")
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/orders"))


# --- OUTPUT B: Order Records ---
# order_shipped_records = (shipped_assignments
#     .withColumn("single_order_id", F.explode(F.split(F.col("order_id"), ", ")))
#     .select(
#         F.col("single_order_id").alias("order_id"),
#         #"customer_id",
#         #order_date
#         F.lit("Shipped").alias("status"),
#         F.col("shipped_time").alias("updated_timestamp"),
#         F.lit(current_run_uuid).alias("_batch_id"),
#         F.current_timestamp().alias("_ingest_timestamp")
#     ).dropDuplicates(["order_id"]))

#display(bin_shipped_records.select("bin_id", "order_id","employee_id", "updated_timestamp", "status"))

#display(order_shipped_records)

In [0]:
%run ./08_silver_post_shipment